# Week 3: Predictive Modeling (Churn Prediction)

This week marks the transition from Exploratory Data Analysis (EDA) to building a robust machine learning pipeline. Our goal is to predict booking cancellations using historical data, allowing the hotel management to take proactive measures.

### Objectives:
1. **Data Preprocessing:** Prepare the dataset by encoding categorical features and scaling numeric ones.
2. **Model Selection:** Implement baseline classification models using Scikit-Learn.
3. **Evaluation:** Use multiple metrics to ensure the model captures the nuances of cancellation patterns.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix, roc_curve, classification_report

# Set aesthetic parameters
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Data Preparation

We load the cleaned dataset produced in Week 1.

In [ ]:
# Load cleaned data
df = pd.read_csv('../WEEK1/cleaned_data.csv')

# Preview features
print(f"Dataset Shape: {df.shape}")
df.head()

### Feature Selection & Preprocessing

Certain variables like `reservationstatus` and `reservationstatusdate` are recorded *after* someone cancels, so they must be removed to prevent data leakage.

In [ ]:
# Drop target-leaking columns and non-predictive dates
cols_to_drop = ['reservationstatus', 'reservationstatusdate', 'country']
df_ml = df.drop(columns=cols_to_drop)

# Identify categorical and numeric columns
cat_cols = df_ml.select_dtypes(include=['object']).columns
num_cols = df_ml.select_dtypes(exclude=['object']).columns.drop('iscanceled')

# Encode categorical variables
le = LabelEncoder()
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

print("Encoding complete. Dataset ready for splitting.")

## 2. Train-Test Split

We split the data into a training set (to teach the model) and a test set (to evaluate its predictions on unseen data).

In [ ]:
X = df_ml.drop('iscanceled', axis=1)
y = df_ml['iscanceled']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training instances: {X_train.shape[0]}")
print(f"Test instances: {X_test.shape[0]}")

# Scale the features for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 3. Modeling - Baseline Algorithms

We compare **Logistic Regression** (probabilistic approach) and **Decision Tree** (rule-based approach).

In [ ]:
# Initialize models
log_reg = LogisticRegression(max_iter=1000)
tree_clf = DecisionTreeClassifier(max_depth=10, random_state=42)

# Train models
log_reg.fit(X_train_scaled, y_train)
tree_clf.fit(X_train, y_train)

print("Model training finished Successfully.")

## 4. Evaluation and Visualization

We compare the models based on project-required metrics: **Accuracy, Precision, Recall, and ROC-AUC**.

In [ ]:
def evaluate_model(model, X_t, y_t, name, is_scaled=False):
    y_pred = model.predict(X_t)
    y_prob = model.predict_proba(X_t)[:, 1]
    
    metrics = {
        'Accuracy': accuracy_score(y_t, y_pred),
        'Precision': precision_score(y_t, y_pred),
        'Recall': recall_score(y_t, y_pred),
        'ROC-AUC': roc_auc_score(y_t, y_prob)
    }
    return metrics, y_prob

results_log, prob_log = evaluate_model(log_reg, X_test_scaled, y_test, 'Logistic Regression')
results_tree, prob_tree = evaluate_model(tree_clf, X_test, y_test, 'Decision Tree')

performance_df = pd.DataFrame([results_log, results_tree], index=['Logistic Regression', 'Decision Tree'])
performance_df

### ROC-AUC Curve

The ROC curve summarizes the trade-off between the true positive rate and false positive rate.

In [ ]:
fpr_log, tpr_log, _ = roc_curve(y_test, prob_log)
fpr_tree, tpr_tree, _ = roc_curve(y_test, prob_tree)

plt.plot(fpr_log, tpr_log, label=f"Logistic Regression (AUC = {results_log['ROC-AUC']:.2f})")
plt.plot(fpr_tree, tpr_tree, label=f"Decision Tree (AUC = {results_tree['ROC-AUC']:.2f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve for Booking Cancellation Prediction')
plt.legend()
plt.show()

### Conclusion

The models show that features like `leadtime`, `deposittype`, and `total_of_special_requests` are significant indicators of potential cancellations. With an AUC score exceeding baseline, the model is fit for generating strategic business insights.